# 🐘 AWS RDS PostgreSQL 실습 — Google Colab

**실습 목표** (PDF 강의안 기반)
1. AWS RDS PostgreSQL 17 인스턴스 자동 생성 (boto3)
2. VPC 보안 그룹 포트 5432 인바운드 규칙 자동 설정
3. psycopg2 로 RDS 연결 확인
4. dvdrental 샘플 데이터베이스 복원
5. 기초 SQL 실습 (테이블 조회, 집계, 조인)

---
**아키텍처**
```
Colab (Python/psycopg2)
    │
    │  port 5432
    ▼
AWS RDS PostgreSQL 17
  (db.t3.micro, 단일 AZ, 퍼블릭 액세스 활성화)
    │
    ▼
dvdrental DB  ← dvdrental.tar 복원
```
---
> ⚠️ **비용 주의**: db.t3.micro는 AWS 프리티어 12개월 무료 (750시간/월).
> 실습 후 반드시 **Cell 10** 의 인스턴스 삭제 코드를 실행하세요.

## Cell 1 — 패키지 설치

In [ ]:
# 필요 라이브러리 설치
!pip install boto3 psycopg2-binary pandas sqlalchemy -q
print('✅ 패키지 설치 완료')

## Cell 2 — AWS 인증 (Colab 비밀 사용)

> **Colab 비밀 설정 방법**
> 1. 왼쪽 사이드바 🔑 (Secrets) 클릭
> 2. 아래 키 이름으로 값 추가:
>    - `AWS_ACCESS_KEY_ID`
>    - `AWS_SECRET_ACCESS_KEY`
>    - `AWS_DEFAULT_REGION` (예: `ap-northeast-2`)
>    - `DB_PASSWORD` (설정할 RDS 마스터 비밀번호 — 8자 이상)

In [ ]:
from google.colab import userdata
import boto3
import time, json, requests

# ── Colab Secrets에서 인증 정보 로드 ──
AWS_ACCESS_KEY_ID     = userdata.get('AWS_ACCESS_KEY_ID').strip()
AWS_SECRET_ACCESS_KEY = userdata.get('AWS_SECRET_ACCESS_KEY').strip()
AWS_REGION            = (userdata.get('AWS_DEFAULT_REGION') or 'ap-northeast-2').strip()
DB_PASSWORD           = userdata.get('DB_PASSWORD').strip()   # 8자 이상, 특수문자 포함

# ── 실습 파라미터 (필요 시 변경) ──
DB_INSTANCE_ID = 'postgresql-lab-01'
DB_NAME        = 'postgres'          # 기본 DB (dvdrental은 별도 생성)
DB_USERNAME    = 'postgres'
DB_PORT        = 5432
DB_CLASS       = 'db.t3.micro'       # 프리티어
DB_ENGINE      = 'postgres'
DB_VERSION     = '17.4'              # PostgreSQL 17.x
STORAGE_GB     = 20

# boto3 세션 구성
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION
)
rds = session.client('rds')
ec2 = session.client('ec2')

print(f'✅ AWS 인증 완료 — 리전: {AWS_REGION}')

def _ensure_aws_session():
    """rds/ec2 클라이언트가 없으면 자동 초기화."""
    g = globals()
    if 'rds' not in g or 'ec2' not in g:
        _sess = boto3.Session(
            aws_access_key_id=g.get('AWS_ACCESS_KEY_ID', userdata.get('AWS_ACCESS_KEY_ID').strip()),
            aws_secret_access_key=g.get('AWS_SECRET_ACCESS_KEY', userdata.get('AWS_SECRET_ACCESS_KEY').strip()),
            region_name=g.get('AWS_REGION', (userdata.get('AWS_DEFAULT_REGION') or 'ap-northeast-2').strip())
        )
        g['rds'] = _sess.client('rds')
        g['ec2'] = _sess.client('ec2')
        print('⚙️  AWS 클라이언트 자동 초기화 완료')

## Cell 3 — 현재 IP 확인 (보안 그룹 인바운드 설정용)

In [ ]:
# Colab 외부 IP 확인 (보안 그룹 인바운드 소스로 사용)
my_ip = requests.get('https://api.ipify.org').text.strip()
MY_CIDR = f'{my_ip}/32'
print(f'🌐 Colab 외부 IP: {my_ip}')
print(f'   → 보안 그룹 인바운드 소스: {MY_CIDR}')

## Cell 4 — VPC 기본 보안 그룹에 PostgreSQL 포트 5432 추가

> PDF p.16~17: 인바운드 규칙 편집 → 포트 5432, 소스 "내 IP" 설정

In [ ]:
_ensure_aws_session()

def setup_security_group_for_postgres(my_cidr: str) -> str:
    """기본 VPC의 default 보안 그룹에 PostgreSQL 5432 인바운드 규칙 추가."""
    vpcs = ec2.describe_vpcs(Filters=[{'Name':'isDefault','Values':['true']}])
    if not vpcs['Vpcs']:
        raise RuntimeError('기본 VPC가 없습니다.')
    vpc_id = vpcs['Vpcs'][0]['VpcId']
    print(f'📌 기본 VPC ID: {vpc_id}')

    sgs = ec2.describe_security_groups(
        Filters=[
            {'Name':'vpc-id',     'Values':[vpc_id]},
            {'Name':'group-name', 'Values':['default']}
        ]
    )
    if not sgs['SecurityGroups']:
        raise RuntimeError(f'VPC({vpc_id})에 default 보안 그룹이 없습니다.')
    sg_id = sgs['SecurityGroups'][0]['GroupId']
    print(f'🔒 보안 그룹 ID: {sg_id}')

    existing = sgs['SecurityGroups'][0].get('IpPermissions', [])
    already = any(
        p.get('FromPort') == 5432 and
        any(r.get('CidrIp') in (my_cidr, '0.0.0.0/0')
            for r in p.get('IpRanges', []))
        for p in existing
    )
    if already:
        print(f'✅ 포트 5432 인바운드 규칙 이미 존재')
    else:
        try:
            ec2.authorize_security_group_ingress(
                GroupId=sg_id,
                IpPermissions=[{
                    'IpProtocol': 'tcp',
                    'FromPort': 5432,
                    'ToPort':   5432,
                    'IpRanges': [{'CidrIp': my_cidr,
                                   'Description': 'PostgreSQL from Colab'}]
                }]
            )
            print(f'✅ 포트 5432 인바운드 규칙 추가 완료 (소스: {my_cidr})')
        except ec2.exceptions.ClientError as e:
            if e.response['Error']['Code'] != 'InvalidPermission.Duplicate':
                raise
            print(f'✅ 포트 5432 인바운드 규칙 이미 존재 (중복 무시)')

    return sg_id

if 'MY_CIDR' not in dir() or MY_CIDR is None:
    import requests as _req
    my_ip = _req.get('https://api.ipify.org').text.strip()
    MY_CIDR = f'{my_ip}/32'
    print(f'🌐 Colab 외부 IP 자동 조회: {MY_CIDR}')

sg_id = setup_security_group_for_postgres(MY_CIDR)
print(f'\n보안 그룹 설정 완료: {sg_id}')

## Cell 5 — AWS RDS PostgreSQL 인스턴스 생성

> PDF p.5~15 기반: 표준 생성, PostgreSQL 17, db.t3.micro, 단일 AZ, 퍼블릭 액세스 활성화

In [ ]:
_ensure_aws_session()

# sg_id 미정의 시 자동 실행
if 'sg_id' not in dir() or sg_id is None:
    print('⚠️  sg_id 미정의 — 보안 그룹 설정 자동 실행...')
    if 'MY_CIDR' not in dir() or MY_CIDR is None:
        import requests as _req
        my_ip = _req.get('https://api.ipify.org').text.strip()
        MY_CIDR = f'{my_ip}/32'
    sg_id = setup_security_group_for_postgres(MY_CIDR)

def create_rds_postgresql(sg_id: str) -> dict:
    """RDS PostgreSQL 인스턴스 생성 (없으면 생성, 있으면 기존 반환)."""
    try:
        resp = rds.describe_db_instances(DBInstanceIdentifier=DB_INSTANCE_ID)
        inst = resp['DBInstances'][0]
        print(f'ℹ️  기존 인스턴스 발견: {DB_INSTANCE_ID} ({inst["DBInstanceStatus"]})')
        return inst
    except rds.exceptions.DBInstanceNotFoundFault:
        pass

    print(f'🚀 RDS PostgreSQL 인스턴스 생성 시작...')
    print(f'   인스턴스 ID : {DB_INSTANCE_ID}')
    print(f'   엔진        : PostgreSQL {DB_VERSION}')
    print(f'   클래스       : {DB_CLASS} (프리티어)')
    print(f'   스토리지     : {STORAGE_GB}GB')
    print(f'   공개 접근    : 활성화 (테스트 환경)')

    rds.create_db_instance(
        DBInstanceIdentifier = DB_INSTANCE_ID,
        DBInstanceClass      = DB_CLASS,
        Engine               = DB_ENGINE,
        EngineVersion        = DB_VERSION,
        MasterUsername       = DB_USERNAME,
        MasterUserPassword   = DB_PASSWORD,
        DBName               = DB_NAME,
        AllocatedStorage     = STORAGE_GB,
        StorageType          = 'gp2',
        PubliclyAccessible   = True,
        MultiAZ              = False,
        AutoMinorVersionUpgrade = True,
        BackupRetentionPeriod   = 1,
        VpcSecurityGroupIds  = [sg_id],
        Tags=[{'Key':'Purpose','Value':'PostgreSQL-Lab'},
              {'Key':'ManagedBy','Value':'Colab-boto3'}]
    )
    print('⏳ 인스턴스 생성 요청 완료 — 사용 가능 상태까지 약 5~10분 대기...')
    return None

inst = create_rds_postgresql(sg_id)

## Cell 6 — RDS 인스턴스 사용 가능 상태 대기 및 엔드포인트 확인

In [ ]:
_ensure_aws_session()

def wait_for_rds_available(instance_id: str, timeout_min: int = 20) -> str:
    """RDS 인스턴스가 'available' 상태가 될 때까지 폴링."""
    print(f'⏳ RDS 상태 대기 중 (최대 {timeout_min}분)...')
    deadline = time.time() + timeout_min * 60
    while time.time() < deadline:
        resp = rds.describe_db_instances(DBInstanceIdentifier=instance_id)
        inst = resp['DBInstances'][0]
        status = inst['DBInstanceStatus']
        print(f'   현재 상태: {status}', end='')

        if status == 'available':
            endpoint = inst['Endpoint']['Address']
            port     = inst['Endpoint']['Port']
            print(f'\n✅ RDS 인스턴스 사용 가능!')
            print(f'   엔드포인트: {endpoint}')
            print(f'   포트       : {port}')
            print(f'   엔진       : {inst["EngineVersion"]}')
            return endpoint
        print(f' — 30초 후 재확인...')
        time.sleep(30)

    raise TimeoutError('RDS 인스턴스 생성 시간 초과')

# ── 인스턴스 상태 확인 ──
print('🔍 AWS RDS PostgreSQL 인스턴스 상태 확인...')
try:
    response = rds.describe_db_instances()
    pg_instances = [i for i in response['DBInstances'] if i['Engine'] == 'postgres']
    if not pg_instances:
        print('ℹ️  현재 실행 중인 PostgreSQL 인스턴스가 없습니다.')
    else:
        print(f'✅ 총 {len(pg_instances)}개의 PostgreSQL 인스턴스 발견:')
        for i in pg_instances:
            print(f'   - ID: {i["DBInstanceIdentifier"]}, 상태: {i["DBInstanceStatus"]}')
except Exception as e:
    print(f'❌ 인스턴스 상태를 가져오는 중 오류 발생: {e}')

RDS_HOST = wait_for_rds_available(DB_INSTANCE_ID)
print(f'\n🔗 연결 정보')
print(f'   Host    : {RDS_HOST}')
print(f'   Port    : {DB_PORT}')
print(f'   User    : {DB_USERNAME}')
print(f'   DB      : {DB_NAME}')

## Cell 7 — psycopg2로 RDS PostgreSQL 연결 확인

> PDF p.21~22: pgAdmin4 또는 psql CLI로 접속 — 여기서는 Python psycopg2 사용

In [ ]:
import psycopg2
from psycopg2.extras import RealDictCursor
import pandas as pd

def get_connection(dbname: str = DB_NAME):
    """RDS PostgreSQL 연결 반환."""
    return psycopg2.connect(
        host     = RDS_HOST,
        port     = DB_PORT,
        dbname   = dbname,
        user     = DB_USERNAME,
        password = DB_PASSWORD,
        connect_timeout = 10
    )

# 연결 테스트
try:
    conn = get_connection()
    cur  = conn.cursor()
    cur.execute('SELECT version();')
    version = cur.fetchone()[0]
    cur.close()
    conn.close()
    print('✅ RDS PostgreSQL 연결 성공!')
    print(f'   버전: {version}')
except Exception as e:
    print(f'❌ 연결 실패: {e}')
    print('   → 보안 그룹 5432 포트 설정 및 엔드포인트를 확인하세요.')

## Cell 8 — dvdrental 데이터베이스 생성 및 복원

> PDF p.23~25: dvdrental DB 생성 → dvdrental.tar 복원

dvdrental은 PostgreSQL 공식 샘플 데이터베이스로 DVD 대여점 비즈니스 모델을 담고 있습니다.
- 15개 테이블 (film, actor, customer, rental, payment, inventory 등)
- 21,000+ 레코드

In [ ]:
import subprocess, os, urllib.request

# ── Step 1: dvdrental.tar 다운로드 ──
DVDRENTAL_URL = 'https://www.postgresqltutorial.com/wp-content/uploads/2019/05/dvdrental.zip'
ZIP_PATH = '/tmp/dvdrental.zip'
TAR_PATH = '/tmp/dvdrental.tar'

print('📥 dvdrental 샘플 DB 다운로드...')
urllib.request.urlretrieve(DVDRENTAL_URL, ZIP_PATH)
subprocess.run(['unzip', '-o', ZIP_PATH, '-d', '/tmp/'], capture_output=True)
print(f'✅ 다운로드 완료: {TAR_PATH}')

# ── Step 2: dvdrental 데이터베이스 생성 ──
print('\n🗄️  dvdrental 데이터베이스 생성...')
conn = get_connection(DB_NAME)
conn.autocommit = True
cur = conn.cursor()

cur.execute("SELECT 1 FROM pg_database WHERE datname = 'dvdrental';")
if cur.fetchone():
    print('ℹ️  dvdrental DB 이미 존재')
else:
    cur.execute('CREATE DATABASE dvdrental OWNER postgres;')
    print('✅ dvdrental 데이터베이스 생성 완료')

cur.close()
conn.close()

# ── Step 3: pg_restore로 dvdrental.tar 복원 ──
print('\n📦 dvdrental.tar 복원 시작...')
!apt-get install -y postgresql-client -q

os.environ['PGPASSWORD'] = DB_PASSWORD
result = subprocess.run(
    ['pg_restore',
     '--host',     RDS_HOST,
     '--port',     str(DB_PORT),
     '--username', DB_USERNAME,
     '--dbname',   'dvdrental',
     '--no-password',
     '--verbose',
     TAR_PATH],
    capture_output=True, text=True
)

if result.returncode == 0:
    print('✅ dvdrental 복원 완료!')
else:
    if 'already exists' in result.stderr or len(result.stderr) < 500:
        print('✅ 복원 완료 (기존 객체 일부 스킵)')
    else:
        print(f'⚠️  복원 출력:\n{result.stderr[:1000]}')

## Cell 9 — dvdrental DB 기초 SQL 실습

> PDF p.26~27: pgAdmin4 주요 메뉴 — 테이블, 뷰, 함수, 집계 실습

In [ ]:
def run_query(sql: str, conn=None) -> pd.DataFrame:
    """SQL 실행 후 DataFrame 반환."""
    _conn = conn or get_connection('dvdrental')
    df = pd.read_sql_query(sql, _conn)
    if not conn:
        _conn.close()
    return df

dv_conn = get_connection('dvdrental')

print('=' * 50)
print('📋 9-1. dvdrental 테이블 목록')
print('=' * 50)
df_tables = run_query("""
    SELECT table_name, table_type
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_type, table_name;
""", dv_conn)
display(df_tables)

In [ ]:
print('=' * 50)
print('🎬 9-2. 영화 목록 (film 테이블 — 상위 10건)')
print('=' * 50)
df_films = run_query("""
    SELECT film_id, title, release_year, rental_rate,
           rating, length AS duration_min
    FROM film
    ORDER BY film_id
    LIMIT 10;
""", dv_conn)
display(df_films)

In [ ]:
print('=' * 50)
print('📊 9-3. 카테고리별 영화 수 (집계 함수)')
print('=' * 50)
df_category = run_query("""
    SELECT c.name AS category,
           COUNT(f.film_id) AS film_count,
           ROUND(AVG(f.rental_rate), 2) AS avg_rental_rate,
           ROUND(AVG(f.length), 1)      AS avg_duration_min
    FROM category c
    JOIN film_category fc ON c.category_id = fc.category_id
    JOIN film f           ON fc.film_id    = f.film_id
    GROUP BY c.name
    ORDER BY film_count DESC;
""", dv_conn)
display(df_category)

In [ ]:
print('=' * 50)
print('👥 9-4. 고객별 대여 횟수 Top 10 (JOIN)')
print('=' * 50)
df_customers = run_query("""
    SELECT c.customer_id,
           c.first_name || ' ' || c.last_name AS full_name,
           c.email,
           COUNT(r.rental_id)      AS total_rentals,
           SUM(p.amount)::numeric(10,2) AS total_paid
    FROM customer c
    JOIN rental  r ON c.customer_id = r.customer_id
    JOIN payment p ON r.rental_id   = p.rental_id
    GROUP BY c.customer_id, c.first_name, c.last_name, c.email
    ORDER BY total_rentals DESC
    LIMIT 10;
""", dv_conn)
display(df_customers)

In [ ]:
print('=' * 50)
print('💰 9-5. 월별 매출 집계 (날짜 함수)')
print('=' * 50)
df_monthly = run_query("""
    SELECT TO_CHAR(payment_date, 'YYYY-MM') AS month,
           COUNT(*)                          AS transactions,
           SUM(amount)::numeric(10,2)        AS total_revenue,
           ROUND(AVG(amount), 2)             AS avg_payment
    FROM payment
    GROUP BY TO_CHAR(payment_date, 'YYYY-MM')
    ORDER BY month;
""", dv_conn)
display(df_monthly)

In [ ]:
print('=' * 50)
print('🎭 9-6. 배우별 출연 영화 수 Top 10 (서브쿼리)')
print('=' * 50)
df_actors = run_query("""
    SELECT a.actor_id,
           a.first_name || ' ' || a.last_name AS actor_name,
           COUNT(fa.film_id) AS film_count
    FROM actor a
    JOIN film_actor fa ON a.actor_id = fa.actor_id
    GROUP BY a.actor_id, a.first_name, a.last_name
    ORDER BY film_count DESC
    LIMIT 10;
""", dv_conn)
display(df_actors)

In [ ]:
print('=' * 50)
print('📦 9-7. 재고 현황 — 대여 가능 / 대여 중 (윈도우 함수)')
print('=' * 50)
df_inventory = run_query("""
    SELECT s.store_id,
           COUNT(DISTINCT i.inventory_id)              AS total_inventory,
           COUNT(DISTINCT CASE
               WHEN r.return_date IS NULL THEN i.inventory_id
           END)                                        AS currently_rented,
           COUNT(DISTINCT CASE
               WHEN r.return_date IS NOT NULL OR r.rental_id IS NULL
               THEN i.inventory_id
           END)                                        AS available
    FROM store s
    JOIN inventory i ON s.store_id = i.store_id
    LEFT JOIN rental r ON i.inventory_id = r.inventory_id
        AND r.return_date IS NULL
    GROUP BY s.store_id
    ORDER BY s.store_id;
""", dv_conn)
display(df_inventory)
dv_conn.close()
print('\n✅ 모든 SQL 실습 완료!')

## Cell 10 — 직접 SQL 실행 (자유 실습)

In [ ]:
MY_SQL = """
SELECT f.title,
       c.name AS category,
       f.rental_rate,
       f.rating
FROM film f
JOIN film_category fc ON f.film_id = fc.film_id
JOIN category c       ON fc.category_id = c.category_id
WHERE f.rental_rate > 4.0
ORDER BY f.rental_rate DESC, f.title
LIMIT 15;
"""

df_result = run_query(MY_SQL)
print(f'📋 결과 행 수: {len(df_result)}')
display(df_result)

## Cell 11 — pgAdmin4 연결 정보 출력

> PDF p.19~21: pgAdmin4에서 AWS RDS PostgreSQL 연결하기

아래 정보를 pgAdmin4에 입력하면 GUI 환경에서도 동일한 DB에 접속할 수 있습니다.

In [ ]:
print('=' * 55)
print('  pgAdmin4 / DBeaver / psql 연결 정보')
print('=' * 55)
print(f'  Host (엔드포인트) : {RDS_HOST}')
print(f'  Port              : {DB_PORT}')
print(f'  Username          : {DB_USERNAME}')
print(f'  Password          : (Colab Secret: DB_PASSWORD)')
print(f'  Database          : dvdrental')
print('=' * 55)
print()
print('psql CLI 명령어:')
print(f'  psql -h {RDS_HOST} -U {DB_USERNAME} -d dvdrental -p {DB_PORT}')
print()
print('SQLAlchemy 연결 문자열:')
print(f'  postgresql://{DB_USERNAME}:<password>@{RDS_HOST}:{DB_PORT}/dvdrental')

## Cell 12 — RDS 인스턴스 삭제 (실습 후 반드시 실행!)

> ⚠️ 실습이 끝나면 불필요한 비용 발생을 막기 위해 인스턴스를 삭제하세요.
> (프리티어라도 750시간 초과 시 요금이 부과됩니다)

In [ ]:
CONFIRM_DELETE = False   # ← True 로 변경 후 실행하면 삭제됩니다

if CONFIRM_DELETE:
    print(f'🗑️  RDS 인스턴스 삭제: {DB_INSTANCE_ID}')
    rds.delete_db_instance(
        DBInstanceIdentifier  = DB_INSTANCE_ID,
        SkipFinalSnapshot     = True,
        DeleteAutomatedBackups = True
    )
    print('⏳ 삭제 중... (약 5분 소요)')
    waiter = rds.get_waiter('db_instance_deleted')
    waiter.wait(DBInstanceIdentifier=DB_INSTANCE_ID,
                WaiterConfig={'Delay': 20, 'MaxAttempts': 30})
    print('✅ RDS 인스턴스 삭제 완료!')
else:
    print('ℹ️  삭제 건너뜀. 삭제하려면 CONFIRM_DELETE = True 로 변경 후 재실행하세요.')
    print(f'   현재 인스턴스: {DB_INSTANCE_ID} — AWS 콘솔에서도 삭제 가능합니다.')

## 📋 실습 요약

| 단계 | 설명 | PDF 참조 |
|------|------|----------|
| Cell 1 | 패키지 설치 (boto3, psycopg2, pandas) | — |
| Cell 2 | Colab Secrets로 AWS 인증 | — |
| Cell 3 | Colab 외부 IP 확인 | p.17 |
| Cell 4 | 보안 그룹 포트 5432 자동 설정 | p.16~17 |
| Cell 5 | RDS PostgreSQL 17 인스턴스 생성 | p.5~15 |
| Cell 6 | 인스턴스 사용 가능 대기 및 엔드포인트 확인 | p.15 |
| Cell 7 | psycopg2 연결 확인 | p.21~22 |
| Cell 8 | dvdrental DB 생성 및 .tar 복원 | p.23~25 |
| Cell 9 | 기초 SQL 실습 (6개 쿼리) | p.26~27 |
| Cell 10 | 자유 SQL 실습 | — |
| Cell 11 | pgAdmin4 연결 정보 출력 | p.19~21 |
| Cell 12 | 인스턴스 삭제 (비용 절약) | — |

---
**dvdrental DB ERD 주요 관계**
```
actor ──< film_actor >── film ──< film_category >── category
                          │
                    inventory
                          │
                       rental ──── customer
                          │
                       payment
```